# Calibración contra datos nulos — UMAP + K-Means (Saber Pro)

Notebook independiente para responder al **punto 1 del Revisor 2**: si la separación entre los ocho perfiles es mejor de lo que se obtendría con datos sin ninguna estructura real.

**La idea:** se generan versiones "revueltas" de tus datos donde cada columna se permuta (revuelve) de forma independiente — se conservan los mismos valores que existen en la realidad, pero se destruye cualquier relación real entre variables (por ejemplo, ya no hay conexión entre horas de trabajo y puntaje). Luego se corre el mismo pipeline (UMAP + K-Means, K=8) sobre esos datos "nulos" y se compara contra los datos reales. Si el pipeline separa igual de bien (o mejor) los datos revueltos que los reales, eso indica que la separación observada podría deberse al procedimiento en sí y no a estructura genuina en los datos.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU.
2. Ajusta la ruta de tu `df_maestra.csv` si no está en `Proyecto/`.
3. `Entorno de ejecución` → `Ejecutar todas`.

La celda 5 corre automáticamente **5 repeticiones sobre datos reales + 10 sobre datos nulos** (15 en total) dentro de un solo bucle — no hace falta repetir la ejecución. Tarda unos 10-15 minutos. El progreso se guarda en tu Drive después de cada repetición, así que si Colab se desconecta, vuelve a correr la misma celda y continúa donde quedó.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import umap.umap_ as umap_cpu
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    df_filtrado = df_limpio[[c for c in cols_usar if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"✅ Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("⚠️  NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"✅ Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler

In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
print(f"Shape X_full: {X_full.shape}")

## 3. Calibración contra datos nulos (automático)

Esta celda corre, **dentro de un solo bucle y en una sola ejecución**:
- 5 repeticiones sobre tus **datos reales** (UMAP+K-Means, K=8),
- 10 repeticiones sobre **datos nulos** (cada columna de `X_full` permutada de forma independiente antes de correr el mismo pipeline).

No hace falta correrla varias veces — el bucle interno ya hace las 15 corridas.

In [ ]:
K = 8
N_EVAL = 50_000
N_FIT = 80_000
N_REAL_REPS = 5
N_NULL_REPS = 10

OUT_DIR = '/content/drive/MyDrive/Proyecto/calibracion_nula'
os.makedirs(OUT_DIR, exist_ok=True)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def metricas(emb, labels):
    sample = min(10_000, len(labels))
    sil = silhouette_score(emb, labels, sample_size=sample, random_state=42)
    ch = calinski_harabasz_score(emb, labels)
    db = davies_bouldin_score(emb, labels)
    return {"silhouette": round(sil, 4), "calinski": round(ch, 2), "davies_bouldin": round(db, 4)}

def permutar_columnas(X, seed):
    rng = np.random.default_rng(seed)
    X_perm = np.empty_like(X)
    for j in range(X.shape[1]):
        X_perm[:, j] = rng.permutation(X[:, j])
    return X_perm

def correr_pipeline(X_source, n_total, seed, fixed_eval_idx=None):
    rng_fit = np.random.default_rng(seed=seed)
    idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
    X_fit = X_source[idx_fit]
    if fixed_eval_idx is None:
        rng_eval = np.random.default_rng(seed=seed + 1000)
        idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)
    else:
        idx_eval = fixed_eval_idx
    X_eval = X_source[idx_eval]

    reducer = umap_cpu.UMAP(n_components=2, random_state=seed, n_neighbors=10,
                             low_memory=True, n_jobs=-1)
    reducer.fit(X_fit)
    emb_eval = reducer.transform(X_eval)

    km = MiniBatchKMeans(n_clusters=K, random_state=seed, n_init='auto', batch_size=10_000)
    labels = km.fit_predict(emb_eval)
    return metricas(emb_eval, labels)

n_total = X_full.shape[0]
rng_eval_real = np.random.default_rng(seed=0)
idx_eval_real = rng_eval_real.choice(n_total, size=N_EVAL, replace=False)

results_path = os.path.join(OUT_DIR, 'resultados.json')
if os.path.exists(results_path):
    results = json.load(open(results_path))
else:
    results = {"real": [], "null": []}

while len(results["real"]) < N_REAL_REPS:
    r = len(results["real"])
    seed = 200 + r
    t_r = time.time()
    m = correr_pipeline(X_full, n_total, seed, fixed_eval_idx=idx_eval_real)
    results["real"].append(m)
    json.dump(results, open(results_path, 'w'), indent=2)
    log(f"[REAL] Repetición {r+1}/{N_REAL_REPS} en {time.time()-t_r:.1f}s -> {m}")

while len(results["null"]) < N_NULL_REPS:
    r = len(results["null"])
    seed = 300 + r
    t_r = time.time()
    X_null = permutar_columnas(X_full, seed=seed + 5000)
    m = correr_pipeline(X_null, n_total, seed, fixed_eval_idx=None)
    results["null"].append(m)
    json.dump(results, open(results_path, 'w'), indent=2)
    log(f"[NULL] Repetición {r+1}/{N_NULL_REPS} en {time.time()-t_r:.1f}s -> {m}")

log("✅ Todas las repeticiones completas.")

## 4. Resultados: real vs. nulo

In [ ]:
real_sil = np.array([m['silhouette'] for m in results['real']])
null_sil = np.array([m['silhouette'] for m in results['null']])
real_db  = np.array([m['davies_bouldin'] for m in results['real']])
null_db  = np.array([m['davies_bouldin'] for m in results['null']])
real_ch  = np.array([m['calinski'] for m in results['real']])
null_ch  = np.array([m['calinski'] for m in results['null']])

p_sil = float((null_sil >= real_sil.mean()).mean())
p_ch  = float((null_ch  >= real_ch.mean()).mean())

log(f"Silhouette: real={real_sil.mean():.4f}±{real_sil.std():.4f}  null={null_sil.mean():.4f}±{null_sil.std():.4f}  p≈{p_sil:.3f}")
log(f"Davies-Bouldin (menor=mejor): real={real_db.mean():.4f}  null={null_db.mean():.4f}")
log(f"Calinski-Harabasz: real={real_ch.mean():.1f}  null={null_ch.mean():.1f}  p≈{p_ch:.3f}")

summary = {
    "real_silhouette_mean": float(real_sil.mean()), "null_silhouette_mean": float(null_sil.mean()),
    "real_davies_bouldin_mean": float(real_db.mean()), "null_davies_bouldin_mean": float(null_db.mean()),
    "real_calinski_mean": float(real_ch.mean()), "null_calinski_mean": float(null_ch.mean()),
    "p_empirico_silhouette": p_sil, "p_empirico_calinski": p_ch,
}
json.dump(summary, open(os.path.join(OUT_DIR, 'summary.json'), 'w'), indent=2)
log(f"✅ Resumen guardado en {OUT_DIR}")

## 5. Figura: datos reales vs. datos nulos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.2), dpi=150)

def boxplot_pair(ax, real, null, titulo, mejor_es_menor=False):
    bp = ax.boxplot([real, null], tick_labels=['Datos reales', 'Datos nulos'],
                     widths=0.5, patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#3B6FA0', '#B3B3B3']):
        patch.set_facecolor(color); patch.set_alpha(0.75)
    for i, vals in enumerate([real, null]):
        x = np.random.default_rng(1).normal(i+1, 0.04, size=len(vals))
        ax.scatter(x, vals, s=18, color='#222', zorder=3)
    ax.set_title(titulo, fontsize=10)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_xlabel('(menor = mejor)' if mejor_es_menor else '(mayor = mejor)', fontsize=8)

boxplot_pair(axes[0], real_sil, null_sil, 'Silhouette')
boxplot_pair(axes[1], real_db, null_db, 'Davies-Bouldin', mejor_es_menor=True)
boxplot_pair(axes[2], real_ch, null_ch, 'Calinski-Harabasz')

plt.suptitle('Pipeline (UMAP+K-Means, K=8): datos reales vs. datos nulos', fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_calibracion_nula.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Figura guardada en Drive")